In [1]:
#  imports and setup
import os, time, json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

# PATHS (adjust if needed)
PROJECT_ROOT = Path("/Users/syedadnanahmad/Downloads/AI_TUMOUR_detection")
EMB_ROOT = PROJECT_ROOT / "embeddings"
IDX_CSV = PROJECT_ROOT / "notebooks" / "slide_index.csv"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# DEVICE
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

# Load index
df_index = pd.read_csv(IDX_CSV)
classes = sorted(df_index["class"].unique())
class_map = {c:i for i,c in enumerate(classes)}
num_classes = len(classes)
print("Classes:", classes)


Device: mps
Classes: ['mdoscc', 'normal', 'osmf', 'pdoscc', 'wdoscc']


In [2]:
# Cell 2: Dataset for slide embeddings
class SlideEmbDataset(Dataset):
    def __init__(self, index_df, split, class_map):
        self.df = index_df[index_df["split"] == split].reset_index(drop=True)
        self.class_map = class_map
        self.items = []

        for _, row in self.df.iterrows():
            slide_id = row["slide_id"]
            cls = row["class"]
            emb_path = EMB_ROOT / split / cls / f"{slide_id}.npy"
            meta_path = EMB_ROOT / split / cls / f"{slide_id}_meta.json"

            if emb_path.exists():
                self.items.append({
                    "emb_path": emb_path,
                    "label": class_map[cls],
                    "slide_id": slide_id,
                    "class": cls,
                    "meta": meta_path
                })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        emb = np.load(it["emb_path"]).astype(np.float32)
        emb = torch.from_numpy(emb)
        label = torch.tensor(it["label"], dtype=torch.long)
        return {"emb": emb, "label": label, "slide_id": it["slide_id"], "class": it["class"]}


In [3]:
# Cell 3: AttentionMIL with Top-K pooling

class AttentionMIL_TopK(nn.Module):
    def __init__(self, emb_dim=512, hidden_dim=256, num_classes=5, k=20):
        super().__init__()
        self.k = k

        # attention network (gated)
        self.V = nn.Linear(emb_dim, hidden_dim)
        self.U = nn.Linear(emb_dim, hidden_dim)
        self.w = nn.Linear(hidden_dim, 1)

        # classifier head
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, emb_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(emb_dim // 2, num_classes)
        )

    def forward(self, H):
        # H: (N, D)
        Vh = torch.tanh(self.V(H))
        Uh = torch.sigmoid(self.U(H))
        A = self.w(Vh * Uh).squeeze(1)        # (N,)

        # Top-K selection BEFORE softmax
        K = min(self.k, H.shape[0])
        top_vals, top_idx = torch.topk(A, K)

        H_top = H[top_idx]                    # (K, D)
        A_top = torch.softmax(top_vals, dim=0)

        agg = torch.sum(A_top.unsqueeze(1) * H_top, dim=0)

        logits = self.classifier(agg)
        return logits.unsqueeze(0), A_top, top_idx


In [4]:
# Cell 4: Prepare Dataloaders
train_ds = SlideEmbDataset(df_index, "train", class_map)
val_ds   = SlideEmbDataset(df_index, "val", class_map)
test_ds  = SlideEmbDataset(df_index, "test", class_map)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=lambda x: x[0])
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x[0])
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x[0])

from collections import Counter
counts = Counter([it["label"] for it in train_ds.items])
class_weights = []
total = sum(counts.values())
for i in range(num_classes):
    class_weights.append(total / (num_classes * counts[i]))

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)


In [8]:
#  training and validation loops

def train_epoch(model, loader, optimizer):
    model.train()
    losses, y_true, y_pred, y_scores = [], [], [], []

    for batch in loader:
        emb = batch["emb"].to(device)
        label = batch["label"].to(device)

        optimizer.zero_grad()
        logits, attn, idx = model(emb)
        loss = criterion(logits, label.unsqueeze(0))
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        probs = F.softmax(logits, dim=1).detach().cpu().numpy()[0]

        y_scores.append(probs)
        y_pred.append(np.argmax(probs))
        y_true.append(label.item())

    return np.mean(losses), np.array(y_true), np.array(y_pred), np.vstack(y_scores)

def validate_epoch(model, loader):
    model.eval()
    losses, y_true, y_pred, y_scores = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            emb = batch["emb"].to(device)
            label = batch["label"].to(device)

            logits, attn, idx = model(emb)
            loss = criterion(logits, label.unsqueeze(0))
            losses.append(loss.item())
            
            probs = F.softmax(logits, dim=1).detach().cpu().numpy()[0]

            y_scores.append(probs)
            y_pred.append(np.argmax(probs))
            y_true.append(label.item())

    return np.mean(losses), np.array(y_true), np.array(y_pred), np.vstack(y_scores)


In [9]:
# train Top-K model

def train_topk(k_value):
    print(f"\n======================")
    print(f" TRAINING TOP-K = {k_value}")
    print(f"======================")

    model = AttentionMIL_TopK(
        emb_dim=512,
        hidden_dim=256,
        num_classes=num_classes,
        k=k_value
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_f1 = -1
    patience = 5
    no_improve = 0

    for epoch in range(1, 25):
        t0 = time.time()
        tr_loss, tr_y, tr_pred, tr_score = train_epoch(model, train_loader, optimizer)
        va_loss, va_y, va_pred, va_score = validate_epoch(model, val_loader)

        _, _, f1, _ = precision_recall_fscore_support(
            va_y, va_pred, labels=range(num_classes), zero_division=0
        )
        macro_f1 = np.mean(f1)

        print(f"Epoch {epoch}: val_macro_f1 = {macro_f1:.4f}")

        scheduler.step(macro_f1)

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            no_improve = 0
            torch.save(model.state_dict(), MODEL_DIR / f"mil_topk{k_value}_best.pth")
            print("Saved best model.")
        else:
            no_improve += 1

        if no_improve >= patience:
            print("Early stopping.")
            break

    return model

# Train both models
model_k20 = train_topk(20)
model_k50 = train_topk(50)



 TRAINING TOP-K = 20
Epoch 1: val_macro_f1 = 0.1767
Saved best model.
Epoch 2: val_macro_f1 = 0.2575
Saved best model.
Epoch 3: val_macro_f1 = 0.2404
Epoch 4: val_macro_f1 = 0.3490
Saved best model.
Epoch 5: val_macro_f1 = 0.2842
Epoch 6: val_macro_f1 = 0.3510
Saved best model.
Epoch 7: val_macro_f1 = 0.3339
Epoch 8: val_macro_f1 = 0.3547
Saved best model.
Epoch 9: val_macro_f1 = 0.3589
Saved best model.
Epoch 10: val_macro_f1 = 0.5085
Saved best model.
Epoch 11: val_macro_f1 = 0.5857
Saved best model.
Epoch 12: val_macro_f1 = 0.5309
Epoch 13: val_macro_f1 = 0.5324
Epoch 14: val_macro_f1 = 0.6225
Saved best model.
Epoch 15: val_macro_f1 = 0.5888
Epoch 16: val_macro_f1 = 0.6182
Epoch 17: val_macro_f1 = 0.6020
Epoch 18: val_macro_f1 = 0.6459
Saved best model.
Epoch 19: val_macro_f1 = 0.6699
Saved best model.
Epoch 20: val_macro_f1 = 0.6633
Epoch 21: val_macro_f1 = 0.6638
Epoch 22: val_macro_f1 = 0.6428
Epoch 23: val_macro_f1 = 0.6460
Epoch 24: val_macro_f1 = 0.6586
Early stopping.

 TRA

In [10]:
# Cell 7: evaluation on test set

def evaluate_model(model_path):
    model = AttentionMIL_TopK(
        emb_dim=512,
        hidden_dim=256,
        num_classes=num_classes,
        k=20  # placeholder; k unused at inference if weights loaded
    ).to(device)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    te_loss, te_y, te_pred, te_score = validate_epoch(model, test_loader)

    print("\n=== Model:", model_path.name, "===")
    print("Test Loss:", te_loss)
    print(classification_report(te_y, te_pred, target_names=classes, zero_division=0))

    cm = confusion_matrix(te_y, te_pred)
    return cm

cm20 = evaluate_model(MODEL_DIR / "mil_topk20_best.pth")
cm50 = evaluate_model(MODEL_DIR / "mil_topk50_best.pth")



=== Model: mil_topk20_best.pth ===
Test Loss: 0.9721715371807417
              precision    recall  f1-score   support

      mdoscc       0.51      0.60      0.55        42
      normal       0.90      0.64      0.75        14
        osmf       0.80      0.65      0.71        31
      pdoscc       0.80      0.42      0.55        19
      wdoscc       0.52      0.66      0.58        44

    accuracy                           0.61       150
   macro avg       0.71      0.59      0.63       150
weighted avg       0.65      0.61      0.61       150


=== Model: mil_topk50_best.pth ===
Test Loss: 0.9456847540537516
              precision    recall  f1-score   support

      mdoscc       0.46      0.71      0.56        42
      normal       1.00      0.64      0.78        14
        osmf       0.77      0.77      0.77        31
      pdoscc       0.75      0.16      0.26        19
      wdoscc       0.59      0.55      0.56        44

    accuracy                           0.60       150